# Train a tiny decoder. Then try to invalidate the result.

Can a model copy a repeating pattern? Train it on CPU, remove access to earlier symbols, and replay training from a checkpoint. Then deliberately break the causal mask and corrupt a **copy** of the checkpoint to see the checks fail.

1. Write your predictions in the next cell.
2. Use **Run all** with a Python kernel that has PyTorch installed.
3. Compare the measured results with your predictions.

No API key, GPU, model weights, or external dataset is needed. This notebook contains its complete lab source; execution makes no network requests. It is a synthetic teaching experiment, not evidence of natural-language model quality.

[Full explanation and resource contract](https://sungeuns.github.io/foundation-model-engineering/chapter-7/tiny-decoder-lab/) · [Source repository](https://github.com/sungeuns/foundation-model-engineering)


**Read the recorded run** (Python 3.12.14 · PyTorch 2.15.0+cpu · CPU)

The code cells ran in order in one kernel. Write down your predictions, then compare them with the saved outputs. The file path in the last cell belongs to this run's temporary folder. [Lesson and starter notebook](https://sungeuns.github.io/foundation-model-engineering/chapter-7/tiny-decoder-lab/)

In [1]:
# Replace the empty strings with your predictions before running the experiment.
predictions = {
    "earlier_logits_change_when_future_tokens_change": "",
    "self_only_attention_helps_or_hurts": "",
    "resumed_training_matches_uninterrupted_training": "",
}
seed, steps = 7, 160


## Setup and scope

Use Python 3.12+ and a Jupyter kernel with PyTorch. If the import below fails, install PyTorch **in that kernel's environment** using its [official installer](https://pytorch.org/get-started/locally/). This notebook does not install or upgrade packages automatically. Verified on 2026-11-18 with Python 3.12 and PyTorch 2.15.0+cpu; other compatible runtimes may run it, but identical trajectories across versions or hardware are not promised.

The fixed recipe is one layer, width 32, two attention heads, FP32 CPU, batch 16, and AdamW (learning-rate scale 0.003, betas 0.9/0.95, epsilon 1e-8, decay 0.01, gradient clipping 1.0). It uses a ten-update warmup multiplied by a cosine envelope; the source contains the exact schedule. The first four symbols are the prompt; each sequence has 13 supervised continuation targets. Rotations of a motif remain in one group before the 128/16/16 train/validation/test split.

Parameters, gradients and Adam moments occupy about 158.8 KiB; that excludes activations, temporary buffers, runtime, checkpoints and evaluation. A 1 GiB free-memory allowance is a planning estimate, not a measured peak. There is no sharding, mixed precision, chat template, padding, remote recovery or distributed execution. Preserve the original recipe for the first run; choosing settings after inspecting test results requires a new held-out set for a new generalization claim.


In [2]:
import hashlib
import importlib.util
import json
import math
from pathlib import Path
import sys
import tempfile
import time
from zipfile import ZIP_DEFLATED, ZipFile

if sys.version_info < (3, 12):
    raise RuntimeError("Use a Python 3.12+ kernel for this teaching lab.")
import torch

print({"python": sys.version.split()[0], "torch": str(torch.__version__), "device": "cpu"})


{'python': '3.12.14', 'torch': '2.15.0+cpu', 'device': 'cpu'}


## The complete source travels with the notebook

The following cell is generated from the book's unchanged <code>public/labs/causal_lab.py</code>. It defines a source string, not a second implementation. Expand it to inspect the model, mask, loss and recovery protocol. The next cell verifies its SHA-256 before importing it. Every run uses a fresh directory beside the notebook; previous results are not overwritten.


In [3]:
SOURCE = r'''"""CPU teaching lab: causal copying, an attention ablation, and exact resume.

Python 3.12+, PyTorch 2.15 CPU. No downloads, credentials, or external data.
Run: python causal_lab.py --output /tmp/causal-lab --steps 160 --seed 7
Not a language model benchmark or a distributed training implementation.
"""
import argparse
import copy
import hashlib
import json
import math
from pathlib import Path
import random
import time

import torch
from torch import nn
from torch.nn import functional as F


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def corpus():
    # Rotations of a motif are one semantic group. Split groups BEFORE expansion.
    rng, groups = random.Random(20261117), set()
    while len(groups) < 160:
        motif = tuple(rng.sample(range(1, 17), 4))
        groups.add(min(motif[i:] + motif[:i] for i in range(4)))
    groups = sorted(groups, key=digest)
    splits = {"train": groups[:128], "validation": groups[128:144], "test": groups[144:]}
    manifest = {"generator": "distinct-4-cycle-v1", "seed": 20261117,
                "tokenizer": "integer-identity-v1; ids 1..16; 0 unused",
                "provenance": "locally generated synthetic symbols; no personal data",
                "filter": "four distinct symbols; canonical cyclic rotation",
                "mixture": {"distinct-4-cycle": 1.0}, "groups": splits}
    tensors = {name: torch.tensor([(m * 5)[:17] for m in motifs])
               for name, motifs in splits.items()}
    return tensors, manifest


class Decoder(nn.Module):
    def __init__(self, attention=True):
        super().__init__()
        self.tokens = nn.Embedding(17, 32)
        self.positions = nn.Embedding(16, 32)
        self.norm1, self.norm2 = nn.LayerNorm(32), nn.LayerNorm(32)
        self.attention = nn.MultiheadAttention(32, 2, dropout=0.0, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(32, 64), nn.GELU(), nn.Linear(64, 32))
        self.output = nn.Linear(32, 17)
        # MultiheadAttention boolean masks use True for BLOCKED positions.
        mask = torch.triu(torch.ones(16, 16, dtype=torch.bool), diagonal=1)
        if not attention:
            mask = ~torch.eye(16, dtype=torch.bool)  # self-only ablation
        self.register_buffer("mask", mask)

    def forward(self, ids):
        length = ids.shape[1]
        x = self.tokens(ids) + self.positions(torch.arange(length))
        h = self.norm1(x)
        x = x + self.attention(h, h, h, attn_mask=self.mask[:length, :length],
                               need_weights=False)[0]
        x = x + self.ffn(self.norm2(x))
        return self.output(x)


def loss_sum(model, batch):
    logits = model(batch[:, :-1])  # [B, 16, 17]
    # Four observed symbols form the prompt; score only the 13 repeat targets.
    logits, labels = logits[:, 3:], batch[:, 4:]
    return F.cross_entropy(logits.reshape(-1, 17), labels.reshape(-1), reduction="sum"), labels.numel()


@torch.no_grad()
def evaluate(model, batch):
    model.eval()
    total, count = loss_sum(model, batch)
    return float(total / count)


@torch.no_grad()
def generate(model, prompt, continuation=13):
    """Greedy continuation sees its own predictions, never future targets."""
    model.eval()
    sequence = prompt.clone()
    for _ in range(continuation):
        prediction = model(sequence)[:, -1].argmax(dim=-1, keepdim=True)
        sequence = torch.cat([sequence, prediction], dim=1)
    return sequence


def setup(seed, attention=True):
    torch.manual_seed(seed)
    model = Decoder(attention)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, betas=(0.9, 0.95),
                                  eps=1e-8, weight_decay=0.01)
    sampler = torch.Generator().manual_seed(seed + 1000)
    return model, optimizer, sampler


def advance(model, optimizer, sampler, data, start, end, total_steps):
    model.train()
    trace = []
    for step in range(start, end):
        ids = torch.randint(len(data), (16,), generator=sampler)
        # Schedule uses the FINAL planned budget, including before interruption.
        lr = 0.003 * min((step + 1) / 10, 1.0) * (0.1 + 0.9 * (1 + math.cos(math.pi * step / total_steps)) / 2)
        for group in optimizer.param_groups:
            group["lr"] = lr
        optimizer.zero_grad(set_to_none=True)
        total, count = loss_sum(model, data[ids])
        loss = total / count
        if not torch.isfinite(loss):
            raise RuntimeError("Abort: nonfinite loss")
        loss.backward()
        norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
        optimizer.step()
        trace.append({"step": step + 1, "loss_nats_per_target": float(loss.detach()),
                      "gradient_norm_before_clip": float(norm), "lr": lr,
                      "batch_ids": ids.tolist(), "target_tokens": count})
    return trace


def save_checkpoint(path, model, optimizer, sampler, step, config, manifest):
    state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
             "torch_rng": torch.get_rng_state(), "sampler_rng": sampler.get_state(),
             "step": step, "target_tokens": step * 16 * 13,
             "data_cursor": "replacement sampler RNG plus global step",
             "config": config, "manifest_hash": digest(manifest),
             "config_hash": digest(config), "scheduler": "formula in advance; step/steps",
             "precision": "CPU float32; no scaler; world size 1"}
    temporary = path.with_suffix(".tmp")
    torch.save(state, temporary)
    temporary.replace(path)
    marker = path.with_suffix(".sha256")
    marker.write_text(hashlib.sha256(path.read_bytes()).hexdigest())


def load_checkpoint(path, config, manifest):
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual != path.with_suffix(".sha256").read_text():
        raise ValueError("Incomplete or corrupt checkpoint")
    state = torch.load(path, weights_only=True)
    if state["config_hash"] != digest(config) or state["manifest_hash"] != digest(manifest):
        raise ValueError("Configuration or corpus changed")
    model, optimizer, sampler = setup(config["seed"])
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    sampler.set_state(state["sampler_rng"])
    torch.set_rng_state(state["torch_rng"])
    return model, optimizer, sampler, state["step"]


def run(output, steps=160, seed=7):
    if steps < 20:
        raise ValueError("Use at least 20 steps; learning quality is budget-dependent")
    output.mkdir(parents=True, exist_ok=True)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    data, manifest = corpus()
    config = {"steps": steps, "seed": seed, "batch": 16, "width": 32, "heads": 2,
              "layers": 1, "dtype": "float32", "device": "cpu",
              "code_hash": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
              "torch_version": str(torch.__version__)}
    (output / "manifest.json").write_text(json.dumps(manifest, indent=2))
    model, optimizer, sampler = setup(seed)
    changed = data["train"][:2, :-1].clone()
    original = model(changed).detach()
    changed[:, 8:] = 1
    assert torch.equal(original[:, :8], model(changed).detach()[:, :8]), "Future leakage"
    initial = evaluate(model, data["validation"])
    start_time = time.perf_counter()
    first = advance(model, optimizer, sampler, data["train"], 0, steps // 2, steps)
    checkpoint = output / "checkpoint.pt"
    save_checkpoint(checkpoint, model, optimizer, sampler, steps // 2, config, manifest)
    tail = advance(model, optimizer, sampler, data["train"], steps // 2, steps, steps)
    expected = copy.deepcopy(model.state_dict())
    resumed, resumed_optimizer, resumed_sampler, cursor = load_checkpoint(checkpoint, config, manifest)
    replay = advance(resumed, resumed_optimizer, resumed_sampler, data["train"], cursor, steps, steps)
    assert tail == replay, "Resume changed batches, loss, or gradients"
    assert all(torch.equal(value, resumed.state_dict()[key]) for key, value in expected.items())
    elapsed = time.perf_counter() - start_time
    ablated, ablated_optimizer, ablated_sampler = setup(seed, attention=False)
    advance(ablated, ablated_optimizer, ablated_sampler, data["train"], 0, steps, steps)
    generated = generate(model, data["test"][:, :4])
    correct = generated[:, 4:] == data["test"][:, 4:]
    result = {"config": config, "parameters": sum(p.numel() for p in model.parameters()),
              "initial_validation_nats": initial,
              "validation_nats": evaluate(model, data["validation"]),
              "test_nats": evaluate(model, data["test"]),
              "self_only_test_nats": evaluate(ablated, data["test"]),
              "greedy_test_token_accuracy": float(correct.float().mean()),
              "greedy_test_sequence_accuracy": float(correct.all(dim=1).float().mean()),
              "example": {"prompt": data["test"][0, :4].tolist(),
                          "generated": generated[0, 4:].tolist(),
                          "expected": data["test"][0, 4:].tolist()},
              "exact_resume": True, "causal_invariance": True,
              "train_and_replay_seconds": elapsed,
              "train_and_replay_target_tokens_per_second": (steps + steps - steps // 2) * 208 / elapsed,
              "nonfinite_count": 0, "mfu": None}
    (output / "trace.json").write_text(json.dumps(first + tail, indent=2))
    (output / "result.json").write_text(json.dumps(result, indent=2))
    return result


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--steps", type=int, default=160)
    parser.add_argument("--seed", type=int, default=7)
    args = parser.parse_args()
    print(json.dumps(run(args.output, args.steps, args.seed), indent=2))
'''


In [4]:
EXPECTED_SOURCE_SHA256 = "d3488b924a81a2cc50a381583903e8756c7b76846d1221cabd63e490bba1ba0c"
assert hashlib.sha256(SOURCE.encode("utf-8")).hexdigest() == EXPECTED_SOURCE_SHA256
run_root = Path(tempfile.mkdtemp(prefix="fme-tiny-decoder-", dir=Path.cwd()))
output = run_root / "outputs"
output.mkdir()
source_path = output / "causal_lab.py"
source_path.write_bytes(SOURCE.encode("utf-8"))
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == EXPECTED_SOURCE_SHA256
spec = importlib.util.spec_from_file_location("fme_causal_lab", source_path)
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)
print({"source_sha256": EXPECTED_SOURCE_SHA256, "output_directory": str(output)})


{'source_sha256': 'd3488b924a81a2cc50a381583903e8756c7b76846d1221cabd63e490bba1ba0c', 'output_directory': '/tmp/fme-notebook-smoke-qrmiu54w/fme-tiny-decoder-19u2jfuh/outputs'}


## Run the original experiment

This trains the causal model, saves and restores a midpoint checkpoint, replays its remaining updates, and trains the self-only attention ablation. The CPU implementation checks causal invariance and exact replay and aborts on nonfinite loss/gradients or failed checks. It records token-normalized loss, gradient norms, learning rate, batch IDs and throughput. No validated hardware FLOP model is supplied, so MFU remains missing rather than zero.


In [5]:
started = time.perf_counter()
result = lab.run(output, steps=steps, seed=seed)
notebook_run_seconds = time.perf_counter() - started
print({"full_experiment_seconds": round(notebook_run_seconds, 3),
       "parameters": result["parameters"], "supervised_training_tokens": steps * 16 * 13})


{'full_experiment_seconds': 1.846, 'parameters': 10161, 'supervised_training_tokens': 33280}


## First ask whether the result is valid

A low loss is not enough. These gates concern this CPU run and source snapshot. They do not prove distributed fault tolerance or cross-version determinism. The full-experiment timing above includes the ablation and evaluations; the original lab's separately named training-and-replay timing excludes them.


In [6]:
assert result["causal_invariance"] is True
assert result["exact_resume"] is True
assert result["nonfinite_count"] == 0
assert result["config"]["code_hash"] == EXPECTED_SOURCE_SHA256
print({key: result[key] for key in ("causal_invariance", "exact_resume", "nonfinite_count", "mfu")})


{'causal_invariance': True, 'exact_resume': True, 'nonfinite_count': 0, 'mfu': None}


## What did access to earlier symbols buy?

The self-only ablation keeps parameter count, initialization, sampling and update budget fixed. Compare its test loss with the causal model's test loss; the intervention changes available context, not model size. A uniform predictor over the 16 valid symbols has loss ln(16) ≈ 2.773 nats per target.

Teacher-forced loss supplies correct preceding symbols. Greedy generation feeds predictions back into the model, so one early error can affect later tokens. Whole-sequence accuracy asks a stricter question than token accuracy. These are observations on the synthetic held-out groups, not LLM benchmark scores.


In [7]:
measurements = {key: result[key] for key in (
    "initial_validation_nats", "validation_nats", "test_nats", "self_only_test_nats",
    "greedy_test_token_accuracy", "greedy_test_sequence_accuracy")}
measurements["uniform_valid_symbol_nats"] = math.log(16)
print(json.dumps(measurements, indent=2))
print(json.dumps(result["example"], indent=2))
trace = json.loads((output / "trace.json").read_text())
for index in (0, len(trace) // 2, len(trace) - 1):
    print({key: trace[index][key] for key in ("step", "loss_nats_per_target", "lr", "gradient_norm_before_clip")})


{
  "initial_validation_nats": 3.0856950283050537,
  "validation_nats": 0.04528987035155296,
  "test_nats": 0.050811074674129486,
  "self_only_test_nats": 2.7041945457458496,
  "greedy_test_token_accuracy": 0.9807692170143127,
  "greedy_test_sequence_accuracy": 0.9375,
  "uniform_valid_symbol_nats": 2.772588722239781
}
{
  "prompt": [
    9,
    14,
    10,
    16
  ],
  "generated": [
    9,
    14,
    10,
    16,
    9,
    14,
    10,
    16,
    9,
    14,
    10,
    16,
    9
  ],
  "expected": [
    9,
    14,
    10,
    16,
    9,
    14,
    10,
    16,
    9,
    14,
    10,
    16,
    9
  ]
}
{'step': 1, 'loss_nats_per_target': 3.1487460136413574, 'lr': 0.00030000000000000003, 'gradient_norm_before_clip': 1.0194655656814575}
{'step': 81, 'loss_nats_per_target': 0.4234147369861603, 'lr': 0.0016500000000000002, 'gradient_norm_before_clip': 0.7720319628715515}
{'step': 160, 'loss_nats_per_target': 0.020919401198625565, 'lr': 0.0003002602253492125, 'gradient_norm_before_clip'

## Break one assumption: let attention see the future

This separate, untrained diagnostic model has the causal mask removed. Change only input positions 8 onward and measure the change in logits at earlier positions. A nonzero change exposes future leakage. It is not a new trained comparison or a better copying model; the original model and saved results remain intact.


In [8]:
diagnostic, _, _ = lab.setup(seed)
diagnostic.eval()
diagnostic.mask.zero_()
data, manifest = lab.corpus()
before = data["train"][:2, :-1].clone()
after = before.clone()
after[:, 8:] = 1
with torch.no_grad():
    leak_delta = float((diagnostic(before)[:, :8] - diagnostic(after)[:, :8]).abs().max())
assert leak_delta > 1e-6, "The diagnostic did not expose leakage; inspect the intervention."
print({"unmasked_earlier_logit_max_change": leak_delta})


{'unmasked_earlier_logit_max_change': 0.1990128457546234}


## Break a checkpoint copy

The saved checkpoint contains model and optimizer state, global step and token count, torch/sampler RNG state, data cursor, schedule and configuration/manifest hashes. Its SHA-256 completion marker detects accidental corruption in this exercise. It does not authenticate an untrusted download. Only load the checkpoint just created by this notebook. Keep the intact packet as the known-good recovery point.


In [9]:
broken_dir = run_root / "intentional-corruption"
broken_dir.mkdir(exist_ok=True)
broken = broken_dir / "checkpoint.pt"
broken.write_bytes((output / "checkpoint.pt").read_bytes() + b"deliberate-corruption")
broken.with_suffix(".sha256").write_text((output / "checkpoint.sha256").read_text())
try:
    lab.load_checkpoint(broken, result["config"], manifest)
except ValueError as error:
    corruption_rejected = True
    print({"corruption_rejected": True, "reason": str(error)})
else:
    raise AssertionError("A corrupt checkpoint was accepted.")
# Confirm that the untouched original still restores.
lab.load_checkpoint(output / "checkpoint.pt", result["config"], manifest)


{'corruption_rejected': True, 'reason': 'Incomplete or corrupt checkpoint'}


(Decoder(
   (tokens): Embedding(17, 32)
   (positions): Embedding(16, 32)
   (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
   (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
   (attention): MultiheadAttention(
     (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
   )
   (ffn): Sequential(
     (0): Linear(in_features=32, out_features=64, bias=True)
     (1): GELU(approximate='none')
     (2): Linear(in_features=64, out_features=32, bias=True)
   )
   (output): Linear(in_features=32, out_features=17, bias=True)
 ),
 AdamW (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.95)
     capturable: False
     decoupled_weight_decay: True
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.0016765054848218483
     maximize: False
     weight_decay: 0.01
 ),
 80)

## Keep evidence, then change one thing

The ZIP contains the exact source, immutable synthetic manifest, checkpoint/marker, training trace and result packet. The deliberately corrupted copy is excluded. Keep it with your predictions; change one setting at a time in a **new** run. Three seeds can reveal sensitivity but do not establish a precise confidence interval. Do not release a modified lab artifact if invariance, integrity or replay checks fail, or if its held-out behavior is unexplained.

The ZIP path is printed below. In a hosted notebook, use the file browser to download it before the runtime is discarded. There are no embedded credentials or downloaded datasets.


In [10]:
evidence = {"predictions": predictions, "full_experiment_seconds": notebook_run_seconds,
            "unmasked_earlier_logit_max_change": leak_delta, "corruption_rejected": corruption_rejected}
(output / "notebook-observations.json").write_text(json.dumps(evidence, indent=2))
archive = run_root / "result-packet.zip"
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as packet:
    for name in ("causal_lab.py", "manifest.json", "checkpoint.pt", "checkpoint.sha256",
                 "trace.json", "result.json", "notebook-observations.json"):
        packet.write(output / name, arcname=f"outputs/{name}")
print({"download_packet": str(archive)})


{'download_packet': '/tmp/fme-notebook-smoke-qrmiu54w/fme-tiny-decoder-19u2jfuh/result-packet.zip'}


## Check your explanation

1. Why does removing the mask invalidate a gain on next-token prediction?
2. Why does exact replay need sampler state as well as model weights?
3. Why can token accuracy look high while several full sequences are wrong?

<details><summary>Compare your answers</summary>
Future inputs reveal information unavailable at generation time. The sampler state preserves the next training batches; model weights alone do not preserve optimizer moments, the schedule or RNG state. Whole-sequence accuracy counts a sequence as wrong after even one incorrect token, and free-running errors can propagate.
</details>

## References

- [The complete bilingual lesson](https://sungeuns.github.io/foundation-model-engineering/chapter-7/tiny-decoder-lab/) — derivation, masks, resource accounting and extension questions.
- [PyTorch reproducibility](https://docs.pytorch.org/docs/2.15/notes/randomness.html) — limits across releases, platforms and devices; checked 2026-11-18.
- [PyTorch serialization](https://docs.pytorch.org/docs/2.15/notes/serialization.html) — state dictionaries and loading semantics; checked 2026-11-18.
